# Hidden Markov Models

**Companion lesson:** https://ml-viz-ruby.vercel.app/courses/graphical-models/03-hidden-markov-models

A from-scratch, runnable implementation of the concepts in the lesson.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive. Changes to this view are not saved.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor'] = '#1a1d27'
plt.rcParams['text.color'] = '#e2e8f0'
plt.rcParams['axes.labelcolor'] = '#e2e8f0'
plt.rcParams['xtick.color'] = '#94a3b8'
plt.rcParams['ytick.color'] = '#94a3b8'
plt.rcParams['axes.edgecolor'] = '#334155'
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.color'] = '#1e293b'
plt.rcParams['figure.figsize'] = (8, 5)
np.random.seed(0)

## Intuition — inferring the hidden from the observed

A **Hidden Markov Model** is a Markov chain you can't see: the hidden state (the weather) evolves with
transition probabilities `A`, and at each step you only observe a noisy emission (what someone did),
governed by `B`. Three classic questions, three dynamic-programming algorithms: **How likely is this
observation sequence?** → the **forward** algorithm. **What's the single most likely hidden path?** →
**Viterbi**. **What's the probability of each state at each time, given everything?** →
**forward–backward**. All three replace an exponential sum over hidden paths with an `O(T·K²)`
recursion — the same trick, three flavors. We build them from scratch and verify against brute-force
enumeration over every possible path.

## Warm-up — plain Markov chain simulation

Before the hidden states arrive, start with the simplest possible model: a **plain Markov chain**, where the state itself is directly observed at every step — no emissions, no inference, just $P(s_{t+1} \mid s_t)$ read off a transition matrix. This is [DML #132 — Simulate Markov Chain Transitions](https://github.com/Open-Deep-ML/DML-OpenProblem/tree/main/questions/132_simulate-markov-chain-transitions): given a transition matrix (rows sum to 1), a starting state, and a number of steps, simulate the sequence of states visited (including the initial one).

Everything below builds on exactly this transition-matrix idea — the HMM just adds a layer of emissions on top, so the state itself becomes hidden and only a noisy observation of it is seen.

In [ ]:
def simulate_markov_chain(transition_matrix, initial_state, num_steps):
    """Simulate a plain (fully observed) Markov chain.

    transition_matrix : (K, K) array where transition_matrix[i] sums to 1.
    initial_state      : int, the starting state index.
    num_steps          : int, number of transitions to simulate.

    Returns an array of length num_steps + 1: the initial state, followed by
    the state visited after each step.
    """
    n_states = transition_matrix.shape[0]
    trajectory = np.empty(num_steps + 1, dtype=int)
    trajectory[0] = initial_state
    state = initial_state

    for step in range(num_steps):
        # TODO(you): draw the next state via
        # np.random.choice(n_states, p=transition_matrix[state])
        state = ...
        trajectory[step + 1] = state

    return trajectory

In [ ]:
# Checks — run me
np.random.seed(42)
tm_a = np.array([[0.8, 0.2], [0.3, 0.7]])
assert list(simulate_markov_chain(tm_a, 0, 3)) == [0, 0, 1, 1], \
    "matches the DML #132 fixture exactly (seed=42)"

np.random.seed(1)
tm_b = np.array([[0.9, 0.1], [0.2, 0.8]])
assert list(simulate_markov_chain(tm_b, 0, 2)) == [0, 0, 0], \
    "matches a second DML #132 fixture (seed=1)"

# Edge case: uniform transition matrix -- every state equally likely at every
# step; this also happens to be a published DML #132 fixture (seed=0)
np.random.seed(0)
tm_uniform = np.array([[0.5, 0.5], [0.5, 0.5]])
assert list(simulate_markov_chain(tm_uniform, 1, 4)) == [1, 1, 1, 1, 1], \
    "uniform transition matrix: matches the DML #132 fixture exactly (seed=0)"

# Edge case: absorbing state -- once you land on state 0 you can never leave,
# so the chain must be all zeros regardless of which seed drives the randomness
tm_absorbing = np.array([[1.0, 0.0], [0.4, 0.6]])
for seed in range(10):
    np.random.seed(seed)
    out = simulate_markov_chain(tm_absorbing, 0, 15)
    assert np.all(out == 0), f"state 0 is absorbing -- seed {seed} should never leave it"

# Edge case: single-state chain -- there's only one place to go, ever
tm_single = np.array([[1.0]])
for seed in range(5):
    np.random.seed(seed)
    out = simulate_markov_chain(tm_single, 0, 6)
    assert out.shape == (7,) and np.all(out == 0), "a 1-state chain always stays in state 0"

# Structural checks that hold no matter which seed is active
np.random.seed(7)
out = simulate_markov_chain(tm_a, 1, 20)
assert out.shape == (21,), "output length must be num_steps + 1"
assert out[0] == 1, "the first entry must be the initial state"
assert set(out.tolist()) <= {0, 1}, "every entry must be a valid state index"

print("✅ Warm-up exercise passed")

<details>
<summary>💡 Show solution</summary>

```python
def simulate_markov_chain(transition_matrix, initial_state, num_steps):
    n_states = transition_matrix.shape[0]
    trajectory = np.empty(num_steps + 1, dtype=int)
    trajectory[0] = initial_state
    state = initial_state

    for step in range(num_steps):
        state = np.random.choice(n_states, p=transition_matrix[state])
        trajectory[step + 1] = state

    return trajectory
```

</details>

**What to notice:** the warm-up chain establishes the Markov property — the next state depends only
on the current one — and simulation frequencies match the transition matrix. An HMM is exactly this
chain with a layer of observational noise on top.

## The weather HMM

Hidden states {Rainy, Sunny}; observations {Walk, Shop, Clean}. We implement the **forward** (likelihood), **Viterbi** (best path) and **backward** algorithms from scratch.

In [ ]:
pi = np.array([0.6, 0.4])                       # P(z_1)
A  = np.array([[0.7, 0.3], [0.4, 0.6]])          # transitions
B  = np.array([[0.1, 0.4, 0.5], [0.6, 0.3, 0.1]])# emissions: rows=state, cols=obs
states = ['Rainy', 'Sunny']; obs_names = ['Walk', 'Shop', 'Clean']
obs = [0, 2, 1, 0]                               # Walk, Clean, Shop, Walk
print('observation sequence:', [obs_names[o] for o in obs])

## Forward algorithm — P(observations)

$\alpha_t(j)=B_{j,x_t}\sum_i \alpha_{t-1}(i)A_{ij}$, in $O(K^2T)$ instead of $K^T$.

In [ ]:
def forward(obs, pi, A, B):
    T, K = len(obs), len(pi)
    alpha = np.zeros((T, K))
    alpha[0] = pi * B[:, obs[0]]
    for t in range(1, T):
        alpha[t] = (alpha[t-1] @ A) * B[:, obs[t]]
    return alpha, alpha[-1].sum()

alpha, likelihood = forward(obs, pi, A, B)
print('P(observations) =', round(likelihood, 6))

**What to notice:** the **forward** recursion carries `α_t(k) = P(obs so far, state k)` and updates it
with one matrix-vector product per step — `(α @ A) * B[:, obs]`. Summing the final `α` gives the total
observation likelihood without ever enumerating the `2⁴ = 16` hidden paths.

## Viterbi — the single most likely hidden path

Same recursion with $\max$ instead of $\sum$, plus back-pointers to reconstruct the path.

In [ ]:
def viterbi(obs, pi, A, B):
    T, K = len(obs), len(pi)
    delta = np.zeros((T, K)); psi = np.zeros((T, K), int)
    delta[0] = np.log(pi) + np.log(B[:, obs[0]])     # log space avoids underflow
    for t in range(1, T):
        for j in range(K):
            scores = delta[t-1] + np.log(A[:, j])
            psi[t, j] = scores.argmax()
            delta[t, j] = scores.max() + np.log(B[j, obs[t]])
    path = [int(delta[-1].argmax())]
    for t in range(T-1, 0, -1):
        path.insert(0, psi[t, path[0]])
    return path, delta[-1].max()

path, logp = viterbi(obs, pi, A, B)
print('most likely weather:', [states[s] for s in path])
print('log-probability of that path:', round(logp, 3))

**What to notice:** Viterbi is the forward recursion with **max in place of sum** (in log-space, to
avoid underflow), plus backpointers `ψ` to recover the winning path. Given "Walk, Clean, Shop, Walk"
it decodes the most plausible weather sequence — the algorithm behind classical speech recognition and
POS tagging.

## The library way — verify against brute-force enumeration

With 4 timesteps and 2 states there are only 16 hidden paths, so we can compute everything the honest,
exponential way and check the dynamic programming: the forward likelihood must equal the **sum** over
all paths, and Viterbi's answer must be the **argmax** path.

In [ ]:
from itertools import product

def path_prob(z, obs):
    p = pi[z[0]] * B[z[0], obs[0]]
    for t in range(1, len(obs)):
        p *= A[z[t-1], z[t]] * B[z[t], obs[t]]
    return p

all_paths = list(product([0, 1], repeat=len(obs)))
probs = np.array([path_prob(z, obs) for z in all_paths])

brute_likelihood = probs.sum()
brute_best = all_paths[probs.argmax()]

print(f'forward likelihood {likelihood:.6f}  ==  brute-force sum {brute_likelihood:.6f}')
assert np.isclose(likelihood, brute_likelihood), "forward must equal the sum over all paths"

vit_path, vit_logp = viterbi(obs, pi, A, B)
print(f'Viterbi path {vit_path}  ==  brute-force argmax {list(brute_best)}')
assert tuple(vit_path) == brute_best, "Viterbi must find the argmax path"
assert np.isclose(np.exp(vit_logp), probs.max()), "and its probability must match"
print('\nforward == sum over paths, Viterbi == argmax path (all 16 enumerated) ✓')

**What to notice:** the forward algorithm's likelihood equals the exact sum over all 16 paths, and
Viterbi's path is exactly the brute-force argmax — the dynamic programs are provably computing the
right quantities, in `O(T·K²)` instead of `O(Kᵀ)`. At `T=100`, brute force would need `2¹⁰⁰` paths;
the recursions still need only 400 multiplications.

## Backward algorithm and posterior state probabilities

Combine forward and backward to get $P(z_t \mid \text{all observations})$ — the smoothed belief about each day's weather.

In [ ]:
def backward(obs, A, B):
    T, K = len(obs), A.shape[0]
    beta = np.zeros((T, K)); beta[-1] = 1
    for t in range(T-2, -1, -1):
        beta[t] = (A * B[:, obs[t+1]] * beta[t+1]).sum(axis=1)
    return beta

beta = backward(obs, A, B)
posterior = alpha * beta
posterior /= posterior.sum(axis=1, keepdims=True)
print('P(state | all obs) per day (Rainy, Sunny):')
for t, o in enumerate(obs):
    print(f'  day {t} ({obs_names[o]:5s}): {np.round(posterior[t], 3)}')

**What to notice:** the **posterior** `P(state_t | all observations)` from forward–backward differs
from Viterbi's single best path: it lets *future* observations revise beliefs about *past* days, and it
can be uncertain (probabilities near 0.5) where Viterbi must commit. Per-step posteriors and the single
best path answer different questions.

## Gotchas & tradeoffs

- **Underflow is real.** Products of probabilities shrink exponentially with `T` — use log-space
  (Viterbi above) or per-step scaling (standard for forward–backward).
- **Viterbi path ≠ sequence of per-step argmaxes.** The most likely *path* can disagree with the most
  likely *state at each time*; pick the algorithm that answers your question.
- **The Markov and stationarity assumptions bite** — one-step memory and time-constant `A`, `B` are
  approximations; regime changes break them.
- **Learning the parameters** (when `A`, `B` are unknown) needs **Baum–Welch** — the EM algorithm from
  the probabilistic-models course, with forward–backward as its E-step.

In [ ]:
# Underflow: the raw forward values shrink exponentially with sequence length
rng = np.random.default_rng(0)
for T in [4, 50, 200, 700]:
    long_obs = rng.integers(0, 3, T)
    _, lik = forward(long_obs, pi, A, B)
    print(f'T={T:>3}: P(observations) = {lik:.3e}' + ('   <- underflowed to 0!' if lik == 0.0 else ''))
print('\n-> beyond a few hundred steps the raw probability underflows float64; real code works in log-space or rescales each alpha')

**What to notice:** the raw likelihood plummets ~half an order of magnitude *per step* — `1e-24` by
`T=50`, and by `T=700` it **underflows float64 to exactly `0.0`**. Every per-step factor is < 1 and the
product dies. This is why production HMM code always works with scaled `α`s or log-probabilities; the
naive textbook recursion is correct math but broken numerics.

## Key takeaways

- **Forward** sums over hidden paths to get the sequence likelihood in $O(K^2T)$.
- **Viterbi** swaps sum for max (in log space) to recover the single best path.
- **Forward × backward** gives smoothed posteriors over each hidden state.
- These same recursions, wrapped in EM, become Baum-Welch for learning the parameters.

---
## ✏️ Your turn

The cells below are **exercise scaffolds**: the concept is recapped, the code outline is set, and `# TODO(you)` marks what you fill in. Run the `assert` cell after each — it passes silently when your answer is right.

### Exercise 1 — The forward algorithm

Implement the forward recursion on a small 2-state HMM:

$$\alpha_1 = \pi \odot B_{:,o_1}, \qquad \alpha_{t} = (\alpha_{t-1} A) \odot B_{:, o_t}, \qquad P(\text{obs}) = \sum_s \alpha_T(s)$$

The checks compare it against **brute-force enumeration over every hidden path** — same number, exponentially less work.

In [ ]:
A = np.array([[0.7, 0.3], [0.4, 0.6]])      # transitions
B = np.array([[0.9, 0.1], [0.2, 0.8]])      # emissions: B[state, observation]
pi0 = np.array([0.5, 0.5])


def forward(obs):
    """P(observation sequence) by the forward algorithm."""
    # TODO(you): initialize alpha with pi0 * B[:, obs[0]]
    alpha = ...

    for o in obs[1:]:
        # TODO(you): the recursion (alpha @ A) * B[:, o]
        alpha = ...

    return alpha.sum()

In [ ]:
# Checks — run me
def brute_force(obs):
    total, n = 0.0, len(obs)
    for path in range(2 ** n):
        states = [(path >> i) & 1 for i in range(n)]
        p = pi0[states[0]] * B[states[0], obs[0]]
        for t in range(1, n):
            p *= A[states[t - 1], states[t]] * B[states[t], obs[t]]
        total += p
    return total

for obs in [[0], [0, 1], [0, 1, 1], [1, 0, 1, 0]]:
    assert abs(forward(obs) - brute_force(obs)) < 1e-12, f"forward must equal path enumeration for {obs}"
assert abs(forward([0]) - (0.5 * 0.9 + 0.5 * 0.2)) < 1e-12, "one observation, by hand"
print("✅ Exercise 1 passed")

<details>
<summary>💡 Show solution</summary>

```python
def forward(obs):
    alpha = pi0 * B[:, obs[0]]
    for o in obs[1:]:
        alpha = (alpha @ A) * B[:, o]
    return alpha.sum()
```

</details>

### Exercise 2 — Viterbi

Swap the forward algorithm's **sum** for a **max** (and remember which predecessor won) and you get the single most likely hidden path. The checks compare your decode against brute force over all paths — they must agree exactly.

In [ ]:
def viterbi(obs):
    """Most likely hidden state path for the observation sequence."""
    delta = pi0 * B[:, obs[0]]
    back = []

    for o in obs[1:]:
        cand = delta[:, None] * A          # cand[i, j] = delta[i] * A[i, j]

        # TODO(you): record the best predecessor of each state (argmax over axis 0)
        back.append(...)

        # TODO(you): the new delta: best incoming score per state, times the emission
        delta = ...

    # Backtrack
    path = [int(np.argmax(delta))]
    for bp in reversed(back):
        path.append(int(bp[path[-1]]))
    return list(reversed(path))

In [ ]:
# Checks — run me
def brute_best(obs):
    n, best, best_states = len(obs), -1.0, None
    for path in range(2 ** n):
        states = [(path >> i) & 1 for i in range(n)]
        p = pi0[states[0]] * B[states[0], obs[0]]
        for t in range(1, n):
            p *= A[states[t - 1], states[t]] * B[states[t], obs[t]]
        if p > best:
            best, best_states = p, states
    return best_states

for obs in [[0, 0], [0, 1, 1], [1, 1, 0, 0]]:
    assert viterbi(obs) == brute_best(obs), f"Viterbi must match brute force for {obs}"
assert viterbi([0, 0, 0]) == [0, 0, 0], "all-state-0 observations -> the all-0 path"
print("✅ Exercise 2 passed")

<details>
<summary>💡 Show solution</summary>

```python
def viterbi(obs):
    delta = pi0 * B[:, obs[0]]
    back = []
    for o in obs[1:]:
        cand = delta[:, None] * A
        back.append(np.argmax(cand, axis=0))
        delta = cand.max(axis=0) * B[:, o]
    path = [int(np.argmax(delta))]
    for bp in reversed(back):
        path.append(int(bp[path[-1]]))
    return list(reversed(path))
```

</details>